# Fuzzy Logic Controlled Anti-Lock Braking System (ABS)
Quarter-car braking simulation comparing **No ABS**, **Bang-bang ABS**, **PID ABS** and **Fuzzy ABS**
on four road surfaces.

**How to use:** click *Runtime → Run all*. Every cell runs in order and the plots appear below each cell.

## Step 1 – Install and import libraries

In [ ]:
!pip install -q scikit-fuzzy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from scipy.interpolate import RegularGridInterpolator
import os
os.makedirs("results", exist_ok=True)

## Step 2 – Parameters (quarter-car model)

In [ ]:
m      = 350.0     # kg, quarter of a 1400 kg car
R      = 0.30      # m, wheel radius
J      = 1.0       # kg·m², wheel inertia
g      = 9.81      # m/s²
v0     = 27.8      # m/s  (100 km/h)
T_max  = 1800.0    # N·m, maximum brake torque
tau    = 0.01      # s, brake actuator time constant
dt     = 1e-4      # s, simulation step
dt_c   = 0.01      # s, controller update period (100 Hz)
v_stop = 1.0       # m/s, simulation ends below this speed
lam_ref = 0.15     # target wheel slip for the ABS controllers

## Step 3 – Tyre–road friction (Burckhardt model)
μ(λ) = c1·(1 − e^(−c2·λ)) − c3·λ

In [ ]:
SURFACES = {
    "Dry asphalt": (1.2801, 23.99, 0.52),
    "Wet asphalt": (0.857, 33.822, 0.347),
    "Snow":        (0.1946, 94.129, 0.0646),
    "Ice":         (0.05, 306.39, 0.0),
}

def mu(lam, c):
    c1, c2, c3 = c
    return c1 * (1 - np.exp(-c2 * lam)) - c3 * lam

lam_axis = np.linspace(0, 1, 500)
plt.figure(figsize=(7, 4))
for name, c in SURFACES.items():
    y = mu(lam_axis, c)
    k = np.argmax(y)
    plt.plot(lam_axis, y, label=f"{name} (peak μ={y[k]:.2f} at λ={lam_axis[k]:.2f})")
plt.axvline(lam_ref, ls="--", c="gray", lw=1)
plt.xlabel("Wheel slip λ"); plt.ylabel("Friction coefficient μ")
plt.title("Tyre–road friction vs wheel slip"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/1_mu_slip_curves.png", dpi=150); plt.show()

## Step 4 – Controllers
Each controller is called every 10 ms with the current slip and returns the brake torque command.

In [ ]:
class NoABS:
    name = "No ABS"
    def reset(self): pass
    def __call__(self, lam, T_now): return T_max

class BangBang:
    name = "Bang-bang ABS"
    def reset(self): self.T = 0.0
    def __call__(self, lam, T_now):
        self.T += -250.0 if lam > lam_ref else 60.0      # release fast, re-apply slowly
        self.T = np.clip(self.T, 0, T_max)
        return self.T

class PID:
    name = "PID ABS"
    Kp, Ki, Kd = 4000.0, 30000.0, 20.0
    def reset(self): self.I = 0.0; self.e_prev = None
    def __call__(self, lam, T_now):
        e = lam_ref - lam
        de = 0.0 if self.e_prev is None else (e - self.e_prev) / dt_c
        self.e_prev = e
        I_new = self.I + e * dt_c
        u = self.Kp * e + self.Ki * I_new + self.Kd * de
        if 0 < u < T_max:            # anti-windup: integrate only when not saturated
            self.I = I_new
        return float(np.clip(u, 0, T_max))

### Fuzzy controller
* Inputs: slip error **e = λ_ref − λ** and its rate **de**, both scaled to [−1, 1]
* Output: change in brake torque **dT**, scaled to [−1, 1]
* 5 triangular membership functions each: NB, NS, Z, PS, PB
* 25 rules, Mamdani min-max inference, centroid defuzzification

In [ ]:
Ke, Kde = 1 / 0.15, 1 / 5.0            # input scaling gains (tuning knobs)
Ku_apply, Ku_release = 150.0, 600.0     # output gains: re-apply gently, release fast (N·m per 10 ms)

u = np.linspace(-1, 1, 201)
e_in  = ctrl.Antecedent(u, "e")
de_in = ctrl.Antecedent(u, "de")
dT_out = ctrl.Consequent(u, "dT", defuzzify_method="centroid")
labels = ["NB", "NS", "Z", "PS", "PB"]
centers = [-1, -0.5, 0, 0.5, 1]
for var in (e_in, de_in, dT_out):
    for lab, c in zip(labels, centers):
        var[lab] = fuzz.trimf(u, [c - 0.5, c, c + 0.5])

# rule table: output index = (e index + de index − 2), limited to 0..4
rule_table = [[labels[int(np.clip(i + j - 2, 0, 4))] for j in range(5)] for i in range(5)]
rules = [ctrl.Rule(e_in[labels[i]] & de_in[labels[j]], dT_out[rule_table[i][j]])
         for i in range(5) for j in range(5)]
sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

print("Rule table (rows = e, columns = de):")
print(pd.DataFrame(rule_table, index=["e=" + l for l in labels], columns=["de=" + l for l in labels]))

# Pre-compute the control surface once, then interpolate (fast, like a real ECU lookup table)
grid = np.linspace(-1, 1, 41)
surface = np.zeros((41, 41))
for i, ev in enumerate(grid):
    for j, dev in enumerate(grid):
        sim.input["e"] = ev; sim.input["de"] = dev; sim.compute()
        surface[i, j] = sim.output["dT"]
fuzzy_lookup = RegularGridInterpolator((grid, grid), surface)

E, DE = np.meshgrid(grid, grid, indexing="ij")
fig = plt.figure(figsize=(7, 5)); ax = fig.add_subplot(projection="3d")
ax.plot_surface(E, DE, surface, cmap="viridis")
ax.set_xlabel("e (normalised)"); ax.set_ylabel("de (normalised)"); ax.set_zlabel("dT (normalised)")
ax.set_title("Fuzzy controller control surface")
plt.tight_layout(); plt.savefig("results/2_fuzzy_control_surface.png", dpi=150); plt.show()

class FuzzyABS:
    name = "Fuzzy ABS"
    def reset(self): self.T = 0.0; self.e_prev = None
    def __call__(self, lam, T_now):
        e = lam_ref - lam
        de = 0.0 if self.e_prev is None else (e - self.e_prev) / dt_c
        self.e_prev = e
        en, den = np.clip(e * Ke, -1, 1), np.clip(de * Kde, -1, 1)
        out = fuzzy_lookup([[en, den]])[0]
        self.T = np.clip(self.T + (Ku_apply if out > 0 else Ku_release) * out, 0, T_max)
        return self.T

## Step 5 – Simulation loop
Slip λ = (v − ωR)/v ; vehicle m·dv/dt = −μ·m·g ; wheel J·dω/dt = μ·m·g·R − T_b ; brake dT_b/dt = (T_cmd − T_b)/τ

In [ ]:
def simulate(controller, c, switch_to=None, t_switch=None):
    controller.reset()
    v, w, x, Tb, t = v0, v0 / R, 0.0, 0.0, 0.0
    T_cmd, steps_per_ctrl = 0.0, int(round(dt_c / dt))
    log, k, surf = [], 0, c
    while v > v_stop and t < 120:
        if switch_to is not None and t >= t_switch:
            surf = switch_to
        lam = (v - w * R) / max(v, 0.1)
        if k % steps_per_ctrl == 0:
            T_cmd = controller(lam, Tb)
        mu_now = mu(lam, surf)
        v  += -mu_now * g * dt
        w  += (mu_now * m * g * R - Tb) / J * dt
        w   = max(w, 0.0)
        Tb += (T_cmd - Tb) / tau * dt
        x  += v * dt
        t  += dt; k += 1
        if k % 10 == 0:
            log.append((t, v, w * R, lam, Tb, x))
    return pd.DataFrame(log, columns=["t", "v", "wR", "slip", "Tb", "x"])

## Step 6 – Run every controller on every surface

In [ ]:
controllers = [NoABS(), BangBang(), PID(), FuzzyABS()]
runs, rows = {}, []
for s_name, c in SURFACES.items():
    for ctl in controllers:
        df = simulate(ctl, c)
        runs[(s_name, ctl.name)] = df
        after = df[df.t > 0.2]
        rows.append({
            "Surface": s_name, "Controller": ctl.name,
            "Stopping distance (m)": round(df.x.iloc[-1], 2),
            "Stopping time (s)": round(df.t.iloc[-1], 2),
            "Slip RMSE": round(float(np.sqrt(np.mean((after.slip - lam_ref) ** 2))), 3),
            "Wheel locked": bool((after.slip > 0.98).any()),
        })
summary = pd.DataFrame(rows)
summary.to_csv("results/summary.csv", index=False)
summary

## Step 7 – Plots

In [ ]:
# Speed plots: No ABS vs Fuzzy ABS on wet asphalt
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, cname in zip(axs, ["No ABS", "Fuzzy ABS"]):
    df = runs[("Wet asphalt", cname)]
    ax.plot(df.t, df.v, label="Vehicle speed v")
    ax.plot(df.t, df.wR, label="Wheel speed ωR")
    ax.set_title(f"{cname} – wet asphalt"); ax.set_xlabel("Time (s)"); ax.grid(alpha=.3); ax.legend()
axs[0].set_ylabel("Speed (m/s)")
plt.tight_layout(); plt.savefig("results/3_speeds_noabs_vs_fuzzy.png", dpi=150); plt.show()

# Slip vs time for all controllers on wet asphalt
plt.figure(figsize=(8, 4))
for ctl in controllers:
    df = runs[("Wet asphalt", ctl.name)]
    plt.plot(df.t, df.slip, label=ctl.name)
plt.axhline(lam_ref, ls="--", c="k", lw=1, label="Target slip")
plt.ylim(0, 1.05); plt.xlabel("Time (s)"); plt.ylabel("Wheel slip λ")
plt.title("Wheel slip – wet asphalt"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/4_slip_wet.png", dpi=150); plt.show()

# Stopping distance bar chart
pivot = summary.pivot(index="Surface", columns="Controller", values="Stopping distance (m)")
pivot = pivot.loc[list(SURFACES)][[c.name for c in controllers]]
pivot.plot.bar(figsize=(9, 4.5), logy=True, rot=0)
plt.ylabel("Stopping distance (m, log scale)"); plt.title("Stopping distance from 100 km/h")
plt.grid(alpha=.3, axis="y"); plt.tight_layout(); plt.savefig("results/5_stopping_distance.png", dpi=150); plt.show()

## Step 8 – Robustness test: road changes from dry to ice at t = 1 s

In [ ]:
plt.figure(figsize=(8, 4))
rows = []
for ctl in controllers:
    df = simulate(ctl, SURFACES["Dry asphalt"], switch_to=SURFACES["Ice"], t_switch=1.0)
    plt.plot(df.t, df.slip, label=ctl.name)
    rows.append({"Controller": ctl.name, "Stopping distance (m)": round(df.x.iloc[-1], 2),
                 "Max slip after switch": round(df[df.t > 1.0].slip.max(), 2),
                 "Wheel locked after switch": bool((df[df.t > 1.0].slip > 0.98).any())})
plt.axvline(1.0, c="gray", ls=":"); plt.axhline(lam_ref, ls="--", c="k", lw=1)
plt.xlim(0, 4); plt.ylim(0, 1.05); plt.xlabel("Time (s)"); plt.ylabel("Wheel slip λ")
plt.title("Dry → ice at t = 1 s"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/6_dry_to_ice.png", dpi=150); plt.show()
pd.DataFrame(rows)

## Step 9 – Download all results
In Colab: run the cell below, then a `results.zip` file downloads to your computer.

In [ ]:
import shutil
shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Not in Colab – results are in the 'results' folder.")